# Statistical analysis

### Hedef değişkenin Sınıf dengesi

In [124]:
df14.describe()  # Sayısal değişkenler için
df14['Exited'].value_counts(normalize=True)  # Sınıf dengesine bak


Exited
0    0.7962
1    0.2038
Name: proportion, dtype: float64

### Sayısal Değişkenler ile Exited Arasındaki İlişki

In [125]:
from scipy.stats import ttest_ind

# Sayısal değişkenler listesi
numerical_columns = ['CreditScore', 'Age', 'Balance', 'EstimatedSalary', 'Point_Earned']

# Sonuçları saklayacağımız bir liste
results = []

# Her bir sayısal değişken için t-testi uygulama
for col in numerical_columns:
    group0 = df14[df14['Exited'] == 0][col]
    group1 = df14[df14['Exited'] == 1][col]
    
    # t-testi
    stat, p = ttest_ind(group0, group1, equal_var=False)
    
    # Sonuçları ekle
    if p < 0.05:
        result = f"{col}: Anlamlı ilişki var (p={p:.4f})"
    else:
        result = f"{col}: Anlamlı ilişki yok (p={p:.4f})"
    
    results.append(result)

# Sonuçları yazdır
for result in results:
    print(result)

CreditScore: Anlamlı ilişki var (p=0.0093)
Age: Anlamlı ilişki var (p=0.0000)
Balance: Anlamlı ilişki var (p=0.0000)
EstimatedSalary: Anlamlı ilişki yok (p=0.2143)
Point_Earned: Anlamlı ilişki yok (p=0.6429)


### Kategorik Değişkenler ile Hedef (Exited) Arasındaki İlişki

In [126]:
from scipy.stats import chi2_contingency

# Kategorik değişkenler listesi
categorical_columns = ['Geography', 'Gender', 'HasCrCard', 'IsActiveMember', 'Card_Type', 'LoyaltySegment', 'RiskSegment', 'EngagementLevel']

# Sonuçları saklayacağımız bir liste
categorical_results = []

# Her bir kategorik değişken için Ki-kare testi uygulama
for col in categorical_columns:
    contingency = pd.crosstab(df14[col], df14['Exited'])  # Çapraz tablo oluştur
    chi2, p, dof, expected = chi2_contingency(contingency)  # Ki-kare testi
    if p < 0.05:
        result = f"{col}: Anlamlı ilişki var (p={p:.4f})"
    else:
        result = f"{col}: Anlamlı ilişki yok (p={p:.4f})"
    
    categorical_results.append(result)

# Sonuçları yazdır
for result in categorical_results:
    print(result)

Geography: Anlamlı ilişki var (p=0.0000)
Gender: Anlamlı ilişki var (p=0.0000)
HasCrCard: Anlamlı ilişki yok (p=0.5026)
IsActiveMember: Anlamlı ilişki var (p=0.0000)
Card_Type: Anlamlı ilişki yok (p=0.1679)
LoyaltySegment: Anlamlı ilişki var (p=0.0000)
RiskSegment: Anlamlı ilişki var (p=0.0000)
EngagementLevel: Anlamlı ilişki var (p=0.0000)


### Multikolinearite Kontrolü (VIF - Variance Inflation Factor)

In [127]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# Sayısal değişkenlerin VIF hesaplaması
X = df14[numerical_columns]
X = add_constant(X)  # Sabit terim ekle

vif_data = pd.DataFrame()
vif_data["Variable"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_data)

          Variable        VIF
0            const  71.642103
1      CreditScore   1.000059
2              Age   1.000880
3          Balance   1.001225
4  EstimatedSalary   1.000226
5     Point_Earned   1.000220


### Hosmer-Lemeshow Testi

In [128]:
import statsmodels.api as sm
from scipy.stats import chi2
import numpy as np

# Modeli kur
X = df14[numerical_columns + categorical_columns]  # Özellikler
y = df14['Exited']  # Hedef değişken
X = sm.add_constant(X)  # Sabit terim ekle

# Lojistik regresyon modeli
model = sm.Logit(y, X)
result = model.fit()

# Predicted probabilities (tahmin edilen olasılıklar)
pred_probs = result.predict(X)

# Hosmer-Lemeshow Testi
def hosmer_lemeshow_test(y_true, y_pred, num_groups=10):
    # Veriyi num_groups sayıda gruba ayır
    groups = np.array([np.percentile(y_pred, i * 100 / num_groups) for i in range(num_groups + 1)])
    
    observed = []
    expected = []
    for i in range(num_groups):
        # Her grup için gözlemler ve tahmin edilen değerler
        group_indices = np.where((y_pred >= groups[i]) & (y_pred < groups[i + 1]))[0]
        observed.append(np.sum(y_true[group_indices]))
        expected.append(np.sum(y_pred[group_indices]))
    
    # Hosmer-Lemeshow testi
    observed = np.array(observed)
    expected = np.array(expected)
    chi2_stat = np.sum((observed - expected)**2 / expected)
    p_value = chi2.sf(chi2_stat, num_groups - 2)  # Dereceyi num_groups - 2 olarak alıyoruz
    return chi2_stat, p_value

# Testi uygula
chi2_stat, p_value = hosmer_lemeshow_test(y, pred_probs)
print(f"Chi2 Statistic: {chi2_stat}")
print(f"p-value: {p_value}")

Optimization terminated successfully.
         Current function value: 0.433041
         Iterations 6
Chi2 Statistic: 13.887747919399793
p-value: 0.08473855197355479
